In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!git clone https://github.com/sandeep-kota/rf-detr.git
%cd rf-detr

fatal: destination path 'rf-detr' already exists and is not an empty directory.
/content/rf-detr


In [5]:
!pip install supervision


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.5/181.5 kB 6.6 MB/s eta 0:00:00


In [6]:
from rfdetr.detr import RFDETRBase
import torch

In [8]:

# Initialize model with segmentation
model = RFDETRBase(
    enable_segmentation=True,
    # pretrain_weights="blue_bins_best.pth",  # Use COCO pretrained weights
    mask_loss_coef=1.0,
    dice_loss_coef=1.0,
    mask_channels=256,
    num_classes=3
)

# model.model.model = model.model.model.to(device)
model.model.reinitialize_detection_head(num_classes=3)
print(model.model.model)


Loading pretrain weights


reinitializing detection head with 90 classes


LWDETR(
  (transformer): Transformer(
    (decoder): TransformerDecoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerDecoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (dropout1): Dropout(p=0, inplace=False)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (cross_attn): MSDeformAttn(
            (sampling_offsets): Linear(in_features=256, out_features=64, bias=True)
            (attention_weights): Linear(in_features=256, out_features=32, bias=True)
            (value_proj): Linear(in_features=256, out_features=256, bias=True)
            (output_proj): Linear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=2048, bias=True)
          (dropout): Dropout(p=0, inplace=False)
          (linear2): Linear(in_features=2048, out_features=256, bias=True

In [11]:

# Train model
model.train(
    # Dataset configuration
    dataset_dir="/content/drive/MyDrive/Colab Drive/coco_format",
    train_split="train",
    val_split="valid",

    # Training parameters
    batch_size=2,
    epochs=5,
    learning_rate=1e-4,
    weight_decay=1e-4,

    # Segmentation parameters
    enable_segmentation=True,
    mask_loss_coef=1.0,
    dice_loss_coef=1.0,

    # Logging and checkpoints
    output_dir="./segmentation_model",
    tensorboard=True,
    save_period=10,

    # Early stopping
    early_stopping=True,
    early_stopping_patience=10,

    # Optional: use mixed precision training
    amp=True
)


TensorBoard logging initialized. To monitor logs, use 'tensorboard --logdir ./segmentation_model' and open http://localhost:6006/ in browser.
Not using distributed mode
git:
  sha: 7275b0b1fa689d54b299a05a8036caa92bae85ef, status: clean, branch: develop

Namespace(num_classes=3, grad_accum_steps=4, amp=True, lr=0.0001, lr_encoder=0.00015, batch_size=2, weight_decay=0.0001, epochs=5, lr_drop=100, clip_max_norm=0.1, lr_vit_layer_decay=0.8, lr_component_decay=0.7, do_benchmark=False, dropout=0, drop_path=0.0, drop_mode='standard', drop_schedule='constant', cutoff_epoch=0, pretrained_encoder=None, pretrain_weights='rf-detr-base.pth', pretrain_exclude_keys=None, pretrain_keys_modify_to_load=None, pretrained_distiller=None, encoder='dinov2_windowed_small', vit_encoder_num_layers=12, window_block_indexes=None, position_embedding='sine', out_feature_indexes=[2, 5, 8, 11], freeze_encoder=False, layer_norm=True, rms_norm=False, backbone_lora=False, force_no_pretrain=False, dec_layers=3, dim_feed

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:

# Export trained model
model.export(
    format="onnx",
    output_path="./segmentation_model/model.onnx",
    input_shape=(3, 640, 640)
)

In [10]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'